In [11]:
import heapq
import math
import numpy as np
from scipy import stats

In [12]:
# ─────────────────────────────────────────────
#  Inter-arrival / service time generators
# ─────────────────────────────────────────────

def gen_exponential(mean):
    return np.random.exponential(mean)

def gen_erlang(mean, k=2):
    """Erlang-k with given mean (each stage has mean/k)."""
    return np.random.gamma(shape=k, scale=mean / k)

def gen_hyperexponential(p1=0.8, lam1=0.8333, p2=0.2, lam2=5.0):
    """Hyperexponential: mix of two exponentials."""
    if np.random.random() < p1:
        return np.random.exponential(1 / lam1)
    else:
        return np.random.exponential(1 / lam2)

def gen_constant(mean):
    return mean

def gen_pareto(mean, k):
    """
    Pareto with shape k and given mean.
    Mean of Pareto(k, x_m) = k*x_m / (k-1), so x_m = mean*(k-1)/k
    Requires k > 1 for finite mean.
    """
    x_m = mean * (k - 1) / k
    return (np.random.pareto(k) + 1) * x_m

In [13]:
# ─────────────────────────────────────────────
#  Core simulation
# ─────────────────────────────────────────────

def run_simulation(n_customers, m, arrival_gen, service_gen):
    """
    Simulate a blocking system (m servers, no queue).

    Returns
    -------
    blocked : list of 0/1 per customer (1 = blocked)
    """
    # Event types
    ARRIVAL   = 0
    DEPARTURE = 1

    clock        = 0.0
    busy_servers = 0          # number of servers currently in use
    events       = []         # min-heap: (time, event_type)
    blocked      = []         # outcome per customer

    # Schedule first arrival
    heapq.heappush(events, (arrival_gen(), ARRIVAL))
    arrivals_processed = 0

    while arrivals_processed < n_customers:
        time, etype = heapq.heappop(events)
        clock = time

        if etype == ARRIVAL:
            arrivals_processed += 1

            if busy_servers < m:
                # Accepted
                busy_servers += 1
                blocked.append(0)
                service_time = service_gen()
                heapq.heappush(events, (clock + service_time, DEPARTURE))
            else:
                # Blocked
                blocked.append(1)

            # Schedule next arrival
            heapq.heappush(events, (clock + arrival_gen(), ARRIVAL))

        elif etype == DEPARTURE:
            busy_servers -= 1

    return blocked


In [14]:
# ─────────────────────────────────────────────
#  Subsampling for confidence intervals
# ─────────────────────────────────────────────

def confidence_interval(blocked, n_batches=10, alpha=0.05):
    """
    Split blocked list into n_batches, compute mean blocking fraction
    per batch, then return overall estimate and 95% CI.
    """
    n = len(blocked)
    batch_size = n // n_batches
    batch_means = []
    for i in range(n_batches):
        batch = blocked[i * batch_size:(i + 1) * batch_size]
        batch_means.append(np.mean(batch))

    overall_mean = np.mean(batch_means)
    se = stats.sem(batch_means)
    t_crit = stats.t.ppf(1 - alpha / 2, df=n_batches - 1)
    ci = (overall_mean - t_crit * se, overall_mean + t_crit * se)
    return overall_mean, ci

In [15]:
# ─────────────────────────────────────────────
#  Erlang-B formula (analytical reference)
# ─────────────────────────────────────────────

def erlang_b(A, m):
    """Erlang's B formula: blocking probability for M/G/m/m."""
    numerator = (A ** m) / math.factorial(m)
    denominator = sum((A ** i) / math.factorial(i) for i in range(m + 1))
    return numerator / denominator

In [16]:
# ─────────────────────────────────────────────
#  Main: run all parts
# ─────────────────────────────────────────────

def run_scenario(label, n_customers, m, arrival_gen, service_gen, n_batches=10):
    blocked = run_simulation(n_customers, m, arrival_gen, service_gen)
    mean, ci = confidence_interval(blocked, n_batches=n_batches)
    print(f"  {label}")
    print(f"    Blocking fraction : {mean:.4f}")
    print(f"    95% CI            : ({ci[0]:.4f}, {ci[1]:.4f})")
    return mean, ci

In [19]:
# ── Parameters ──────────────────────────────
np.random.seed(42)
m                = 10       # number of servers
mean_service     = 8.0      # mean service time
mean_interarrival = 1.0     # mean inter-arrival time  → A = 8 Erlang
n_runs           = 10       # sub-simulations (batches of customers)
n_per_run        = 10_000   # customers per batch
n_total          = n_runs * n_per_run

A = mean_service / mean_interarrival
B_exact = erlang_b(A, m)

print(f"Blocking system: m={m}, A={A} Erlang")
print(f"Erlang-B (exact analytical): {B_exact:.4f}")

results = {}

Blocking system: m=10, A=8.0 Erlang
Erlang-B (exact analytical): 0.1217


In [23]:
# ── Part 1: Poisson arrivals, Exponential service ────────────
print("Part 1 , Poisson arrivals, Exponential service")
mean, ci = run_scenario(
    "Exp inter-arrivals, Exp service",
    n_total, m,
    arrival_gen  = lambda: gen_exponential(mean_interarrival),
    service_gen  = lambda: gen_exponential(mean_service),
)
results["Part1"] = (mean, ci)

Part 1 , Poisson arrivals, Exponential service
  Exp inter-arrivals, Exp service
    Blocking fraction : 0.1184
    95% CI            : (0.1146, 0.1222)


In [22]:
# ── Part 2a: Erlang inter-arrivals ───────────────────────────
print("Part 2a , Erlang inter-arrivals (k=2), Exponential service")
mean, ci = run_scenario(
    "Erlang-2 inter-arrivals, Exp service",
    n_total, m,
    arrival_gen  = lambda: gen_erlang(mean_interarrival, k=2),
    service_gen  = lambda: gen_exponential(mean_service),
)
results["Part2a"] = (mean, ci)

Part 2a , Erlang inter-arrivals (k=2), Exponential service
  Erlang-2 inter-arrivals, Exp service
    Blocking fraction : 0.0946
    95% CI            : (0.0908, 0.0984)


In [24]:
# ── Part 2b: Hyperexponential inter-arrivals ─────────────────
print("Part 2b , Hyperexponential inter-arrivals, Exponential service")
mean, ci = run_scenario(
    "Hyperexp inter-arrivals (p1=0.8,λ1=0.8333,p2=0.2,λ2=5.0), Exp service",
    n_total, m,
    arrival_gen  = lambda: gen_hyperexponential(),
    service_gen  = lambda: gen_exponential(mean_service),
)
results["Part2b"] = (mean, ci)

Part 2b , Hyperexponential inter-arrivals, Exponential service
  Hyperexp inter-arrivals (p1=0.8,λ1=0.8333,p2=0.2,λ2=5.0), Exp service
    Blocking fraction : 0.1385
    95% CI            : (0.1326, 0.1445)


In [26]:
# ── Part 3a: Poisson arrivals, Constant service ──────────────
print("Part 3a , Poisson arrivals, Constant service")
mean, ci = run_scenario(
    "Exp inter-arrivals, Constant service",
    n_total, m,
    arrival_gen  = lambda: gen_exponential(mean_interarrival),
    service_gen  = lambda: gen_constant(mean_service),
)
results["Part3a"] = (mean, ci)

Part 3a , Poisson arrivals, Constant service
  Exp inter-arrivals, Constant service
    Blocking fraction : 0.1246
    95% CI            : (0.1227, 0.1266)


In [27]:
# ── Part 3b: Poisson arrivals, Pareto service (k=1.05) ───────
print("Part 3b , Poisson arrivals, Pareto service (k=1.05)")
mean, ci = run_scenario(
    "Exp inter-arrivals, Pareto service (k=1.05)",
    n_total, m,
    arrival_gen  = lambda: gen_exponential(mean_interarrival),
    service_gen  = lambda: gen_pareto(mean_service, k=1.05),
)
results["Part3b_k105"] = (mean, ci)

Part 3b , Poisson arrivals, Pareto service (k=1.05)
  Exp inter-arrivals, Pareto service (k=1.05)
    Blocking fraction : 0.0043
    95% CI            : (0.0014, 0.0072)


In [28]:
# ── Part 3b: Poisson arrivals, Pareto service (k=2.05) ───────
print("Part 3c , Poisson arrivals, Pareto service (k=2.05)")
mean, ci = run_scenario(
    "Exp inter-arrivals, Pareto service (k=2.05)",
    n_total, m,
    arrival_gen  = lambda: gen_exponential(mean_interarrival),
    service_gen  = lambda: gen_pareto(mean_service, k=2.05),
)
results["Part3b_k205"] = (mean, ci)

Part 3c , Poisson arrivals, Pareto service (k=2.05)
  Exp inter-arrivals, Pareto service (k=2.05)
    Blocking fraction : 0.1240
    95% CI            : (0.1181, 0.1298)


In [29]:
# ── Part 3c: Poisson arrivals, Erlang service (k=2) ─────────
print("Part 3d , Poisson arrivals, Erlang service (k=2)")
mean, ci = run_scenario(
    "Exp inter-arrivals, Erlang-2 service",
    n_total, m,
    arrival_gen  = lambda: gen_exponential(mean_interarrival),
    service_gen  = lambda: gen_erlang(mean_service, k=2),
)
results["Part3c"] = (mean, ci)

Part 3d , Poisson arrivals, Erlang service (k=2)
  Exp inter-arrivals, Erlang-2 service
    Blocking fraction : 0.1206
    95% CI            : (0.1165, 0.1247)


In [ ]:
# ── Part 4: Summary table ────────────────────────────────────
print("\n" + "=" * 60)
print("Part 4 , Summary comparison")
print(f"  Erlang-B exact: {B_exact:.4f}")
print("-" * 60)
labels = {
    "Part1"       : "Poisson / Exponential service",
    "Part2a"      : "Erlang-2 arrivals / Exp service",
    "Part2b"      : "Hyperexp arrivals / Exp service",
    "Part3a"      : "Poisson / Constant service",
    "Part3b_k105" : "Poisson / Pareto (k=1.05)",
    "Part3b_k205" : "Poisson / Pareto (k=2.05)",
    "Part3c"      : "Poisson / Erlang-2 service",
}
for key, label in labels.items():
    m_val, ci_val = results[key]
    print(f"  {label:<40s}  {m_val:.4f}  CI=({ci_val[0]:.4f}, {ci_val[1]:.4f})")
print("=" * 60)


Part 4 , Summary comparison
  Erlang-B exact: 0.1217
------------------------------------------------------------
  Poisson / Exponential service             0.1184  CI=(0.1146, 0.1222)
  Erlang-2 arrivals / Exp service           0.0946  CI=(0.0908, 0.0984)
  Hyperexp arrivals / Exp service           0.1385  CI=(0.1326, 0.1445)
  Poisson / Constant service                0.1246  CI=(0.1227, 0.1266)
  Poisson / Pareto (k=1.05)                 0.0043  CI=(0.0014, 0.0072)
  Poisson / Pareto (k=2.05)                 0.1240  CI=(0.1181, 0.1298)
  Poisson / Erlang-2 service                0.1206  CI=(0.1165, 0.1247)
